# Practice: Build a Coding Scheme and Test It

**Computational Text Analysis Pack · Module 17 — Annotation and Agreement**

*Technique:* From category definitions to a measured agreement figure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/17c_practice_coding_scheme.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **You want to hand-code 500 documents for a category of your own devising.
> How do you know the scheme works before you spend three weeks on it?**

The answer is a pilot: code fifty, measure agreement, fix the scheme, repeat.
This notebook simulates the loop so the mechanics are clear, then tells you how
to run it for real.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
# A scheme with a deliberately fuzzy boundary, which is what pilots expose.
SCHEME = {
 "STATE_ACTION": "government, ministry or official body does something",
 "COMMUNITY": "action or experience of a non-state group or ordinary people",
 "BOTH": "state acting on or with a community",
 "NEITHER": "neither is the subject",
}
print("Coding scheme, round 1:\n")
for k, v in SCHEME.items():
    print(f"   {k:<14}{v}")
print()
print("The BOTH category is the trap. Watch what it does to agreement.")

In [ ]:
# Simulate two coders on a pilot of 50. In your project these are real people.
rng = np.random.default_rng(5)
pilot = df.sample(n=50, random_state=7).reset_index(drop=True)
cats = list(SCHEME)

# Coder A applies the scheme; coder B agrees except on the fuzzy boundary.
coder_a = [cats[i % len(cats)] for i in rng.integers(0, len(cats), len(pilot))]
coder_b = []
for lab in coder_a:
    r = rng.random()
    if lab == "BOTH" and r < 0.6:
        coder_b.append(rng.choice(["STATE_ACTION", "COMMUNITY"]))
    elif r < 0.85:
        coder_b.append(lab)
    else:
        coder_b.append(rng.choice(cats))

def kappa(a, b):
    n = len(a); labs = sorted(set(a) | set(b))
    po = sum(x == y for x, y in zip(a, b)) / n
    pa, pb = Counter(a), Counter(b)
    pe = sum((pa[c]/n)*(pb[c]/n) for c in labs)
    return (po - pe) / (1 - pe), po

k1, po1 = kappa(coder_a, coder_b)
print(f"round 1: raw agreement {po1:.3f}, Cohen's kappa {k1:.3f}")
print("Landis & Koch: <0.20 slight, 0.21-0.40 fair, 0.41-0.60 moderate,")
print("0.61-0.80 substantial, >0.80 almost perfect.")

In [ ]:
# Diagnose: WHERE do they disagree? That points at the scheme, not the coders.
mat = pd.DataFrame(0, index=cats, columns=cats)
for a, b in zip(coder_a, coder_b):
    mat.loc[a, b] += 1
print("rows = coder A, columns = coder B\n")
print(mat.to_string())
off = [(a, b, int(mat.loc[a, b])) for a in cats for b in cats if a != b]
off.sort(key=lambda t: -t[2])
print("\nbiggest disagreements:")
for a, b, n in off[:3]:
    print(f"   {a} vs {b}: {n}")
print()
print("A systematic confusion between two categories is a boundary problem in")
print("the manual. Fix the definition, do not retrain the coder.")

In [ ]:
# Round 2: drop the fuzzy category, forcing a decision with a priority rule.
SCHEME2 = {k: v for k, v in SCHEME.items() if k != "BOTH"}
SCHEME2["STATE_ACTION"] += " (takes priority if both apply)"
a2 = [x if x != "BOTH" else "STATE_ACTION" for x in coder_a]
b2 = [x if x != "BOTH" else "STATE_ACTION" for x in coder_b]
k2, po2 = kappa(a2, b2)
print("Round 2 scheme:\n")
for k, v in SCHEME2.items():
    print(f"   {k:<14}{v}")
print(f"\nround 2: raw agreement {po2:.3f}, Cohen's kappa {k2:.3f}")
print(f"change  : kappa {k2-k1:+.3f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(["round 1", "round 2"], [k1, k2], color=["indianred", "seagreen"])
ax.axhline(0.67, ls="--", color="grey", label="Krippendorff's tentative floor")
ax.set_ylabel("kappa"); ax.legend(); ax.set_title("Fixing the scheme, not the coders")
plt.tight_layout()

## How to run this for real

1. **Write definitions with examples and non-examples.** A category without a
   boundary case in the manual will be coded inconsistently.
2. **Pilot on 50 items with two coders working independently.** Independently is
   the whole point; a coder who has seen the other's labels tells you nothing.
3. **Compute kappa or alpha, and read the confusion matrix.** The matrix is more
   useful than the number.
4. **Revise the scheme, not the coders.** If two careful readers disagree, the
   definition is ambiguous.
5. **Repeat until agreement is acceptable, then code the rest.** Report the
   final figure, the number of pilot items, and how many rounds it took.

**Defensible:** "Two coders independently coded a 50-item pilot; Cohen's kappa
was [X] after two revisions of the scheme, which is reported with the codebook."

**Not defensible:** hand-coded categories with no agreement figure. Module 15d
showed independent dictionaries overlapping about half the time — assume your
categories do the same until you have measured otherwise.

### 🔧 Your turn 1

Add a fifth category to the round-1 scheme and re-run. Does kappa go up or down, and why?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Kappa usually falls, for two separate reasons worth distinguishing. More
categories means more opportunities to disagree — a genuine effect. But kappa's
chance-correction also shifts with the number and balance of categories, so part
of the movement is arithmetic rather than substantive. This is exactly why
module 17b's Krippendorff's alpha exists and why comparing kappa across studies
with different schemes is not meaningful.

</details>